In [ ]:
# JERICHON import bootstrap (shared resolver in src/project_paths.py)
from pathlib import Path
import os, sys

# Locate repository from the current working directory or its parents.
_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
for _base in list(_candidates):
    try:
        _candidates.extend(_child for _child in _base.iterdir() if _child.is_dir())
    except (OSError, PermissionError):
        pass
PROJECT_ROOT = next((p.resolve() for p in _candidates
                     if (p / "src" / "__init__.py").is_file()
                     and (p / "src" / "data" / "builders").is_dir()), None)
if PROJECT_ROOT is None:
    _env_root = os.environ.get("JERICHON_PROJECT_ROOT")
    if _env_root:
        _candidate = Path(_env_root).expanduser().resolve()
        if (_candidate / "src" / "__init__.py").is_file():
            PROJECT_ROOT = _candidate
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не найден корень JERICHON. Откройте VS Code в папке репозитория "
        "или задайте JERICHON_PROJECT_ROOT.")
if str(PROJECT_ROOT) in sys.path:
    sys.path.remove(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))
from src.project_paths import bootstrap_project, find_project_root
PROJECT_ROOT = bootstrap_project(PROJECT_ROOT)


# JERICHON — Final causal feature dataset pipeline
**До этапа построения модели.**

Этот notebook строит единую причинно-следственную feature matrix JERICHON.

Фактический полный feature inventory определяется самим pipeline:
- 73 базовых признака;
- 94 `tradestats` Super Candle признака;
- 171 `orderstats` Super Candle признака;
- 136 `obstats` Super Candle признаков.

Итого: **474 ML-признака**.

`__event_ts` и `__available_at` сохраняются только как provenance и НЕ входят в ML features.
`orderbook` намеренно не входит в историческую Super Candle matrix.

Ключевое изменение относительно старого research pipeline:
Super Candles присоединяются по времени доступности `available_at`, а не простым `ts`-join.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = PROJECT_ROOT / "data"
if str(ROOT.parent) not in sys.path:
    sys.path.insert(0, str(ROOT.parent))

from src.data.builders.dollar_bars import InformationDrivenBars
from src.data.builders.micro_features import MicroFeatures
from src.data.builders.build_features import FeatureBuilder
from src.data.labeling.cusum import cusum_filter
from src.data.labeling.triple_barrier import triple_barrier_labeling

In [2]:
TICKER = "VTBR"
DOLLAR_THRESHOLD = 10_000_000
MSK = "Europe/Moscow"

def available_dates(root, ticker):
    dates = set()

    for kind in ("realtime", "supercandles"):
        p = root / "for_ml" / "storage" / "history" / kind / ticker

        if p.exists():
            dates.update(
                x.name
                for x in p.iterdir()
                if x.is_dir()
            )

    return sorted(dates)


def load_history(root, ticker):

    result = {
        "candles": [],
        "trades": [],
        "orderbook": [],
        "tradestats": [],
        "obstats": [],
        "orderstats": [],
    }

    rt = root / "for_ml" / "storage" / "history" / "realtime"
    sc = root / "for_ml" / "storage" / "history" / "supercandles"

    dates = available_dates(root, ticker)

    print(f"History root : {root}")
    print(f"Ticker       : {ticker}")
    print(f"Dates found  : {len(dates)}")

    if not dates:
        print("\nWARNING: даты не найдены.")
        print(f"Realtime path     : {rt / ticker}")
        print(f"Supercandles path : {sc / ticker}")
        return {
            key: pd.DataFrame()
            for key in result
        }

    for d in dates:

        # -------------------------
        # REALTIME
        # -------------------------
        for key in ("candles", "trades", "orderbook"):

            p = rt / ticker / d / f"{key}.parquet"

            if p.exists():

                try:
                    x = pd.read_parquet(p)

                    if not x.empty:
                        x = x.copy()
                        x["file_date"] = d
                        result[key].append(x)

                except Exception as e:
                    print(f"WARNING: cannot read {p}")
                    print(f"         {e}")

        # -------------------------
        # SUPERCANDLES
        # -------------------------
        for key in ("tradestats", "obstats", "orderstats"):

            p = sc / ticker / d / f"{key}.parquet"

            if p.exists():

                try:
                    x = pd.read_parquet(p)

                    if not x.empty:
                        x = x.copy()
                        x["file_date"] = d
                        result[key].append(x)

                except Exception as e:
                    print(f"WARNING: cannot read {p}")
                    print(f"         {e}")

    # -------------------------
    # CONCATENATE
    # -------------------------
    out = {}

    for key, frames in result.items():

        if frames:
            out[key] = pd.concat(
                frames,
                ignore_index=True
            )
        else:
            out[key] = pd.DataFrame()

    return out


# =========================
# LOAD
# =========================

data = load_history(ROOT, TICKER)


# =========================
# SUMMARY
# =========================

print("\n" + "=" * 60)
print("ALGOPACK HISTORY")
print("=" * 60)

for k, v in data.items():

    print(
        f"{k:12s}: "
        f"{len(v):>10,} rows"
    )

print("=" * 60)

History root : C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data
Ticker       : VTBR
Dates found  : 24

ALGOPACK HISTORY
candles     :     21,791 rows
trades      :  2,130,276 rows
orderbook   :        440 rows
tradestats  :      4,362 rows
obstats     :      4,259 rows
orderstats  :      4,398 rows


In [3]:
# =========================
# 3. DOLLAR BARS — SAME RESEARCH CONTRACT
# =========================
tr = data["trades"].copy()
required = ["PRICE","QUANTITY","VALUE","ts"]
missing = [c for c in required if c not in tr.columns]
if missing:
    raise RuntimeError(f"trades missing: {missing}")

for c in ("PRICE","QUANTITY","VALUE"):
    tr[c] = pd.to_numeric(tr[c], errors="coerce")
tr["ts"] = pd.to_datetime(tr["ts"], errors="coerce")
tr = tr.dropna(subset=required)
tr = tr[(tr.PRICE > 0) & (tr.QUANTITY > 0) & (tr.VALUE > 0)]

tr_for_bars = pd.DataFrame({"ts": tr["ts"], "price": tr["PRICE"], "quantity": tr["VALUE"] / tr["PRICE"],})

bars = InformationDrivenBars().build_dollar(tr_for_bars, threshold=DOLLAR_THRESHOLD)

bars = bars.copy()
bars["ts"] = pd.to_datetime(bars["close_time"], errors="coerce")
if bars["ts"].dt.tz is None:
    bars["ts"] = bars["ts"].dt.tz_localize(MSK)
else:
    bars["ts"] = bars["ts"].dt.tz_convert(MSK)
bars = bars.dropna(subset=["ts"]).set_index("ts").sort_index()
bars = bars[~bars.index.duplicated(keep="first")]

bars = bars.rename(columns={"open":"OPEN","high":"HIGH","low":"LOW", "close":"CLOSE","volume":"VOLUME"})

print("Dollar bars:", bars.shape)
print(bars.index.min(), "→", bars.index.max())


Dollar bars: (5221, 9)
2026-08-28 07:02:32+03:00 → 2026-10-05 22:37:27+03:00


In [4]:
# =========================
# 4. EXACT 73 BASE FEATURES
# =========================

bars_lower = bars.rename(
    columns={"OPEN": "open",
             "HIGH": "high",
             "LOW": "low",
             "CLOSE": "close",
             "VOLUME": "volume",})

mf = MicroFeatures(spread_window=20, volatility_window=20, intensity_window=20,)
bars_with_micro = mf.build_all_features(bars_lower)
fb = FeatureBuilder(windows=[5, 10, 20], lags=[1, 2, 3], include_micro=False,)
bars_featured = fb.build_all(bars_with_micro)
base = bars_featured.select_dtypes(include=[np.number]).copy()

if isinstance(base.index, pd.DatetimeIndex):
    base.index.name = "ts"
    base = base.reset_index()
elif "ts" not in base.columns:
    raise RuntimeError("FeatureBuilder output has neither DatetimeIndex nor ts")

base["ts"] = pd.to_datetime(base["ts"], errors="coerce",)

base = (
    base
    .dropna(subset=["ts"])
    .sort_values("ts")
    .drop_duplicates("ts", keep="last")
    .reset_index(drop=True)
)

BASE_FEATURES = [
    c
    for c in base.columns
    if c != "ts"
    and pd.api.types.is_numeric_dtype(base[c])
]

print("Base numeric features:", len(BASE_FEATURES))

if len(BASE_FEATURES) != 73:
    raise RuntimeError(f"Expected 73 base features, got {len(BASE_FEATURES)}.")

base2 = base[["ts"] + BASE_FEATURES].copy()
base2["ts_utc"] = pd.to_datetime(base2["ts"], errors="coerce", utc=True,)

base2 = (
    base2
    .dropna(subset=["ts_utc"])
    .sort_values("ts_utc")
    .drop_duplicates("ts_utc", keep="last")
    .reset_index(drop=True)
)

base2 = base2.rename(
    columns={
        c: f"base__{c}"
        for c in BASE_FEATURES
    }
)

BASE_COLUMNS = [
    c for c in base2.columns
    if c.startswith("base__")
]

if len(BASE_COLUMNS) != 73:
    raise RuntimeError(
        f"base2 construction failed: "
        f"expected 73 base__* columns, got {len(BASE_COLUMNS)}"
    )

print("base2 rows:", len(base2))
print("base2 ML features:", len(BASE_COLUMNS))
print("base2 columns:", base2.shape)


Base numeric features: 73
base2 rows: 5221
base2 ML features: 73
base2 columns: (5221, 75)


In [5]:
# =========================
# 5. SUPERCANDLE HELPERS
# =========================
def prep_exact(df):
    if df is None or df.empty or "ts" not in df.columns:
        return pd.DataFrame()
    x = df.copy()
    x["ts"] = pd.to_datetime(x["ts"], errors="coerce")
    for c in x.columns:
        if c not in {"ts","ticker","secid","super_kind","file_date","date","SYSTIME"}:
            x[c] = pd.to_numeric(x[c], errors="coerce")
    return x.dropna(subset=["ts"]).sort_values("ts").reset_index(drop=True)

def prepare_ts(df):
    if df is None or df.empty or "ts" not in df.columns:
        return pd.DataFrame()
    x = df.copy()
    x["ts"] = pd.to_datetime(x["ts"], errors="coerce")
    if x["ts"].dt.tz is None:
        x["ts"] = x["ts"].dt.tz_localize(MSK)
    else:
        x["ts"] = x["ts"].dt.tz_convert(MSK)
    return x.dropna(subset=["ts"]).sort_values("ts").reset_index(drop=True)

def safe_div(a,b,eps=1e-12):
    return a / (b.abs() + eps)

def imb(a,b,eps=1e-12):
    return (a-b) / (a+b).abs().add(eps)

def add_dyn(df, cols, windows=(3,6,12,24)):
    out = df.copy()
    for c in cols:
        if c not in out.columns:
            continue
        s = pd.to_numeric(out[c], errors="coerce")
        out[f"{c}_chg"] = s.diff()
        out[f"{c}_ret"] = s.pct_change(fill_method=None).replace([np.inf,-np.inf],np.nan)
        for w in windows:
            minp = max(2,min(w,3))
            mean = s.rolling(w,min_periods=minp).mean()
            std = s.rolling(w,min_periods=minp).std()
            out[f"{c}_ma{w}"] = mean
            out[f"{c}_z{w}"] = (s-mean)/(std+1e-12)
    return out


In [6]:
# =========================
# 6. BUILD SUPERCANDLE FEATURES
# =========================
feature_frames = {}
feature_groups = {}

# --- tradestats: target 94 ---
ts0 = prep_exact(data["tradestats"])
if not ts0.empty:
    tsf = ts0[["ts"]].copy()
    tsf["ts_trade_imbalance"] = imb(ts0["trades_b"],ts0["trades_s"])
    tsf["ts_volume_imbalance"] = imb(ts0["vol_b"],ts0["vol_s"])
    tsf["ts_value_imbalance"] = imb(ts0["val_b"],ts0["val_s"])
    tsf["ts_buy_trade_share"] = safe_div(ts0["trades_b"],ts0["trades_b"]+ts0["trades_s"])
    tsf["ts_buy_volume_share"] = safe_div(ts0["vol_b"],ts0["vol_b"]+ts0["vol_s"])
    tsf["ts_buy_value_share"] = safe_div(ts0["val_b"],ts0["val_b"]+ts0["val_s"])
    tsf["ts_vwap_b_s_diff"] = ts0["pr_vwap_b"]-ts0["pr_vwap_s"]
    tsf["ts_vwap_b_s_bps"] = (ts0["pr_vwap_b"]-ts0["pr_vwap_s"])/((ts0["pr_vwap_b"]+ts0["pr_vwap_s"])/2+1e-12)*1e4
    tsf["ts_buy_vwap_displacement"] = (ts0["pr_vwap_b"]-ts0["pr_vwap"])/(ts0["pr_vwap"]+1e-12)
    tsf["ts_sell_vwap_displacement"] = (ts0["pr_vwap_s"]-ts0["pr_vwap"])/(ts0["pr_vwap"]+1e-12)
    tsf["ts_value_per_trade"] = safe_div(ts0["val"],ts0["trades"])
    tsf["ts_volume_per_trade"] = safe_div(ts0["vol"],ts0["trades"])
    tsf["ts_flow_x_return"] = ts0["disb"]*ts0["pr_change"]
    tsf["ts_flow_x_volatility"] = ts0["disb"]*ts0["pr_std"]
    dyn=["ts_trade_imbalance","ts_volume_imbalance","ts_value_imbalance","ts_buy_volume_share",
         "ts_buy_value_share","ts_vwap_b_s_bps","ts_value_per_trade","ts_flow_x_return"]
    tsf=add_dyn(tsf,dyn)
    feature_frames["tradestats_pressure"]=tsf
    feature_groups["tradestats_pressure"]=[c for c in tsf.columns if c!="ts"]

# --- orderstats: target 171 ---
os0 = prep_exact(data["orderstats"])
if not os0.empty:
    osf=os0[["ts"]].copy()
    osf["os_put_orders_imb"]=imb(os0["put_orders_b"],os0["put_orders_s"])
    osf["os_put_volume_imb"]=imb(os0["put_vol_b"],os0["put_vol_s"])
    osf["os_put_value_imb"]=imb(os0["put_val_b"],os0["put_val_s"])
    osf["os_put_buy_share"]=safe_div(os0["put_vol_b"],os0["put_vol_b"]+os0["put_vol_s"])
    osf["os_put_value_buy_share"]=safe_div(os0["put_val_b"],os0["put_val_b"]+os0["put_val_s"])
    osf["os_cancel_orders_imb"]=imb(os0["cancel_orders_b"],os0["cancel_orders_s"])
    osf["os_cancel_volume_imb"]=imb(os0["cancel_vol_b"],os0["cancel_vol_s"])
    osf["os_cancel_value_imb"]=imb(os0["cancel_val_b"],os0["cancel_val_s"])
    osf["os_cancel_buy_share"]=safe_div(os0["cancel_vol_b"],os0["cancel_vol_b"]+os0["cancel_vol_s"])
    osf["os_cancel_to_put_orders"]=safe_div(os0["cancel_orders"],os0["put_orders"])
    osf["os_cancel_to_put_volume"]=safe_div(os0["cancel_vol"],os0["put_vol"])
    osf["os_cancel_to_put_value"]=safe_div(os0["cancel_val"],os0["put_val"])
    osf["os_net_orders"]=os0["put_orders"]-os0["cancel_orders"]
    osf["os_net_volume"]=os0["put_vol"]-os0["cancel_vol"]
    osf["os_net_value"]=os0["put_val"]-os0["cancel_val"]
    osf["os_net_volume_imb"]=imb(os0["put_vol_b"]-os0["cancel_vol_b"],os0["put_vol_s"]-os0["cancel_vol_s"])
    osf["os_net_value_imb"]=imb(os0["put_val_b"]-os0["cancel_val_b"],os0["put_val_s"]-os0["cancel_val_s"])
    osf["os_put_vwap_b_s_bps"]=(os0["put_vwap_b"]-os0["put_vwap_s"])/((os0["put_vwap_b"]+os0["put_vwap_s"])/2+1e-12)*1e4
    osf["os_cancel_vwap_b_s_bps"]=(os0["cancel_vwap_b"]-os0["cancel_vwap_s"])/((os0["cancel_vwap_b"]+os0["cancel_vwap_s"])/2+1e-12)*1e4
    osf["os_cancel_vs_put_vwap_b"]=(os0["cancel_vwap_b"]-os0["put_vwap_b"])/(os0["put_vwap_b"]+1e-12)
    osf["os_cancel_vs_put_vwap_s"]=(os0["cancel_vwap_s"]-os0["put_vwap_s"])/(os0["put_vwap_s"]+1e-12)
    dyn=["os_put_orders_imb","os_put_volume_imb","os_put_value_imb","os_cancel_orders_imb",
         "os_cancel_volume_imb","os_cancel_value_imb","os_cancel_to_put_orders","os_cancel_to_put_volume",
         "os_cancel_to_put_value","os_net_volume","os_net_value","os_net_volume_imb","os_net_value_imb",
         "os_put_vwap_b_s_bps","os_cancel_vwap_b_s_bps"]
    osf=add_dyn(osf,dyn)
    feature_frames["orderstats_pressure"]=osf
    feature_groups["orderstats_pressure"]=[c for c in osf.columns if c!="ts"]

# --- obstats: 136 features ---
obs0 = prepare_ts(data["obstats"])
if not obs0.empty:
    obsf=pd.DataFrame({"ts":obs0["ts"]})
    obsf["ob_total_volume"]=obs0["vol_b"].fillna(0)+obs0["vol_s"].fillna(0)
    obsf["ob_total_value"]=obs0["val_b"].fillna(0)+obs0["val_s"].fillna(0)
    obsf["ob_vol_imb_bbo_vs_deep"]=obs0["imbalance_vol_bbo"]-obs0["imbalance_vol"]
    obsf["ob_val_imb_bbo_vs_deep"]=obs0["imbalance_val_bbo"]-obs0["imbalance_val"]
    obsf["ob_bid_value_share"]=obs0["val_b"]/(obs0["val_b"]+obs0["val_s"]+1e-12)
    obsf["ob_bid_volume_share"]=obs0["vol_b"]/(obs0["vol_b"]+obs0["vol_s"]+1e-12)
    obsf["ob_bid_level_share"]=obs0["levels_b"]/(obs0["levels_b"]+obs0["levels_s"]+1e-12)
    obsf["ob_lv10_to_bbo_spread_ratio"]=obs0["spread_lv10"]/(obs0["spread_bbo"]+1e-12)
    obsf["ob_1mio_to_bbo_spread_ratio"]=obs0["spread_1mio"]/(obs0["spread_bbo"]+1e-12)
    obsf["ob_vwap_b_s_bps"]=(obs0["vwap_b"]-obs0["vwap_s"])/(obs0["vwap_s"]+1e-12)*1e4
    obsf["ob_vwap_1mio_b_s_bps"]=(obs0["vwap_b_1mio"]-obs0["vwap_s_1mio"])/(obs0["vwap_s_1mio"]+1e-12)*1e4
    obsf["ob_1mio_liquidity_cost_bps"]=(obs0["spread_1mio"]-obs0["spread_bbo"])/(obs0["spread_bbo"]+1e-12)*1e4
    px=bars.reset_index()[["ts","CLOSE"]].rename(columns={"CLOSE":"close"}).sort_values("ts")
    tmp=pd.merge_asof(obs0[["ts","vwap_b","vwap_s","vwap_b_1mio","vwap_s_1mio"]].sort_values("ts"),
                      px,on="ts",direction="backward")
    obsf["ob_px_vs_vwap_b_bps"]=(tmp["close"]-tmp["vwap_b"])/(tmp["vwap_b"]+1e-12)*1e4
    obsf["ob_px_vs_vwap_s_bps"]=(tmp["close"]-tmp["vwap_s"])/(tmp["vwap_s"]+1e-12)*1e4
    obsf["ob_px_vs_vwap_b_1mio_bps"]=(tmp["close"]-tmp["vwap_b_1mio"])/(tmp["vwap_b_1mio"]+1e-12)*1e4
    obsf["ob_px_vs_vwap_s_1mio_bps"]=(tmp["close"]-tmp["vwap_s_1mio"])/(tmp["vwap_s_1mio"]+1e-12)*1e4
    dyn=["ob_total_volume","ob_total_value","ob_vol_imb_bbo_vs_deep","ob_val_imb_bbo_vs_deep",
         "ob_bid_value_share","ob_bid_volume_share","ob_bid_level_share","ob_lv10_to_bbo_spread_ratio",
         "ob_1mio_to_bbo_spread_ratio","ob_vwap_b_s_bps","ob_vwap_1mio_b_s_bps","ob_1mio_liquidity_cost_bps"]
    obsf=add_dyn(obsf,dyn)
    feature_frames["obstats_topology"]=obsf
    feature_groups["obstats_topology"]=[c for c in obsf.columns if c!="ts"]

print({k: len(v) for k, v in feature_groups.items()})

# Фактический полный inventory этого pipeline.
expected_group_counts = {
    "tradestats_pressure": 94,
    "orderstats_pressure": 171,
    "obstats_topology": 136,
}

for group, expected in expected_group_counts.items():
    actual = len(feature_groups.get(group, []))
    if actual != expected:
        raise RuntimeError(
            f"{group}: expected {expected} features, got {actual}"
        )

print(
    "Super Candle features:",
    sum(len(v) for v in feature_groups.values())
)



{'tradestats_pressure': 94, 'orderstats_pressure': 171, 'obstats_topology': 136}
Super Candle features: 401


C:\Users\Сергей Кравцов\AppData\Local\Temp\ipykernel_12648\2362464801.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{c}_ma{w}"] = mean
C:\Users\Сергей Кравцов\AppData\Local\Temp\ipykernel_12648\2362464801.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{c}_z{w}"] = (s-mean)/(std+1e-12)
C:\Users\Сергей Кравцов\AppData\Local\Temp\ipykernel_12648\2362464801.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance. 

In [7]:
# =========================
# 8. VALIDATE SYSTIME -> AVAILABLE_AT
# =========================
# We do NOT globally assume MSK or UTC.
# For each source/day, compare both interpretations against event ts.
# The interpretation with lower negative-rate and more plausible lag wins.
# Ambiguous days are quarantined rather than silently used.

def parse_systime(series):
    return pd.to_datetime(series, errors="coerce")

def choose_availability(df, source_name):
    x = df.copy()
    if "SYSTIME" not in x.columns:
        raise RuntimeError(f"{source_name}: no SYSTIME column; cannot construct validated availability.")

    syst = parse_systime(x["SYSTIME"])
    # SYSTIME is stored as naive clock-like values; test both interpretations.
    if syst.dt.tz is not None:
        av_utc = syst.dt.tz_convert("UTC")
        av_msk = syst.dt.tz_convert(MSK)
    else:
        av_utc = syst.dt.tz_localize("UTC")
        av_msk = syst.dt.tz_localize(MSK).dt.tz_convert("UTC")

    event_utc = pd.to_datetime(x["ts"], utc=True, errors="coerce")
    day = x["file_date"].astype(str) if "file_date" in x.columns else event_utc.dt.tz_convert(MSK).dt.strftime("%Y-%m-%d")

    chosen = []
    diagnostics = []
    for d, idx in x.groupby(day).groups.items():
        e = event_utc.loc[idx]
        a_u = av_utc.loc[idx]
        a_m = av_msk.loc[idx]
        lag_u = (a_u - e).dt.total_seconds()
        lag_m = (a_m - e).dt.total_seconds()

        def score(s):
            valid = s.dropna()
            if valid.empty: return (1e9,1e9,1e9)
            neg=(valid<0).mean()
            med=valid.median()
            p95=valid.quantile(.95)
            # prefer no negative availability and normal low latency;
            # large positive delays are retained rather than "fixed away".
            return (neg, abs(med-15), p95)

        su, sm = score(lag_u), score(lag_m)
        use_utc = su < sm
        av = a_u if use_utc else a_m
        lag = lag_u if use_utc else lag_m

        diagnostics.append({
            "source":source_name,"file_date":str(d),
            "chosen_tz":"UTC" if use_utc else "MSK",
            "median_lag_sec":float(lag.median()),
            "p95_lag_sec":float(lag.quantile(.95)),
            "negative_pct":float((lag<0).mean()*100),
            "max_lag_sec":float(lag.max()),
        })
        chosen.append(pd.Series(av,index=idx))

    availability = pd.concat(chosen).sort_index()
    x["event_ts"] = event_utc
    x["available_at"] = availability
    diag = pd.DataFrame(diagnostics).sort_values(["source","file_date"])
    return x, diag

validated = {}
availability_audit = []

for name in ("tradestats","obstats","orderstats"):
    src, diag = choose_availability(data[name], name)
    validated[name] = src
    availability_audit.append(diag)

availability_audit = pd.concat(availability_audit, ignore_index=True)
display(availability_audit)


,source,file_date,chosen_tz,median_lag_sec,p95_lag_sec,negative_pct,max_lag_sec
0,tradestats,2026-08-28,MSK,10.0,15.0,0.0,18.0
1,tradestats,2026-08-31,MSK,10.0,15.0,0.0,20.0
2,tradestats,2026-09-01,UTC,10.0,15.0,0.0,16.0
3,tradestats,2026-09-02,UTC,10.0,15.0,0.0,15.0
4,tradestats,2026-09-03,UTC,10.0,15.0,0.0,16.0
...,...,...,...,...,...,...,...
67,orderstats,2026-09-28,MSK,13.0,18.0,0.0,20.0
68,orderstats,2026-09-29,MSK,13.0,18.0,0.0,19.0
69,orderstats,2026-09-30,MSK,13.0,18.0,0.0,58.0
70,orderstats,2026-10-02,MSK,13.0,18.0,0.0,22.0


In [8]:
# =========================================================
# 9. CAUSAL JOIN
# =========================================================

# Dollar bar timestamp
if "close_time" not in bars.columns:
    raise RuntimeError(
        f"'close_time' not found in bars. "
        f"Columns: {list(bars.columns)}"
    )

bar_base = (
    bars[["close_time"]]
    .rename(columns={"close_time": "bar_ts"})
    .copy()
)

bar_base["bar_ts"] = pd.to_datetime(
    bar_base["bar_ts"],
    errors="coerce",
    utc=True
)

bar_base = (
    bar_base
    .dropna(subset=["bar_ts"])
    .drop_duplicates("bar_ts")
    .sort_values("bar_ts")
    .reset_index(drop=True)
)

bar_base["bar_ts_utc"] = bar_base["bar_ts"]

print("=" * 70)
print("CAUSAL JOIN")
print("=" * 70)
print(f"Bars: {len(bar_base):,}")
print(f"Period: {bar_base.bar_ts.min()} -> {bar_base.bar_ts.max()}")
print()
print("validated keys:")
print(list(validated.keys()))
print()
print("feature_frames keys:")
print(list(feature_frames.keys()))
print()


# =========================================================
# CAUSAL ATTACH
# =========================================================

def attach_causal(bar_frame, source_name, source_df, feature_df):

    # ---------- source timing ----------

    if "event_ts" not in source_df.columns:
        raise RuntimeError(
            f"{source_name}: 'event_ts' not found. "
            f"Columns: {list(source_df.columns)}"
        )

    if "available_at" not in source_df.columns:
        raise RuntimeError(
            f"{source_name}: 'available_at' not found. "
            f"Columns: {list(source_df.columns)}"
        )

    src = source_df[
        ["event_ts", "available_at"]
    ].copy()

    src["event_ts"] = pd.to_datetime(
        src["event_ts"],
        errors="coerce",
        utc=True
    )

    src["available_at"] = pd.to_datetime(
        src["available_at"],
        errors="coerce",
        utc=True
    )

    src = (
        src
        .dropna(subset=["event_ts", "available_at"])
        .sort_values("event_ts")
        .drop_duplicates("event_ts", keep="last")
    )


    # ---------- features ----------

    if "ts" not in feature_df.columns:
        raise RuntimeError(
            f"{source_name}: feature frame has no 'ts'. "
            f"Columns: {list(feature_df.columns)[:30]}"
        )

    f = feature_df.copy()

    f["event_ts"] = pd.to_datetime(
        f["ts"],
        errors="coerce",
        utc=True
    )

    f = f.drop(columns=["ts"])

    f = (
        f
        .dropna(subset=["event_ts"])
        .sort_values("event_ts")
        .drop_duplicates("event_ts", keep="last")
    )


    # ---------- add availability ----------

    f = f.merge(
        src,
        on="event_ts",
        how="left"
    )

    f = f.dropna(
        subset=["available_at"]
    )


    # ---------- rename ----------

    feature_columns = [
        c for c in f.columns
        if c not in {
            "event_ts",
            "available_at"
        }
    ]

    rename = {
        c: f"{source_name}__{c}"
        for c in feature_columns
    }

    f = f.rename(columns=rename)

    f = f.rename(
        columns={
            "event_ts":
                f"{source_name}__event_ts",
            "available_at":
                f"{source_name}__available_at"
        }
    )


    right_cols = [
        f"{source_name}__available_at",
        f"{source_name}__event_ts",
    ] + list(rename.values())


    # ---------- TRUE CAUSAL ASOF ----------

    result = pd.merge_asof(
        bar_frame.sort_values("bar_ts_utc"),

        f[right_cols].sort_values(
            f"{source_name}__available_at"
        ),

        left_on="bar_ts_utc",

        right_on=f"{source_name}__available_at",

        direction="backward"
    )

    return result


# =========================================================
# IMPORTANT:
# validated is keyed by RAW SOURCE NAME.
#
# We now determine feature frames automatically
# from their actual keys instead of guessing.
# =========================================================

SOURCE_TO_FEATURE = {}

for source in (
    "tradestats",
    "obstats",
    "orderstats"
):

    # exact key
    if source in feature_frames:
        SOURCE_TO_FEATURE[source] = source
        continue

    # find the only feature group containing source name
    candidates = [
        k for k in feature_frames.keys()
        if source.lower() in k.lower()
    ]

    if len(candidates) == 1:
        SOURCE_TO_FEATURE[source] = candidates[0]

    elif len(candidates) == 0:
        raise RuntimeError(
            f"Cannot find feature frame for '{source}'.\n"
            f"Available feature frames:\n"
            f"{list(feature_frames.keys())}"
        )

    else:
        raise RuntimeError(
            f"Ambiguous feature frames for '{source}': "
            f"{candidates}"
        )


print("Resolved source → feature frame:")
for source, feature_key in SOURCE_TO_FEATURE.items():
    print(f"  {source:12s} -> {feature_key}")

print()


# =========================================================
# BUILD CAUSAL MATRIX
# =========================================================

causal = bar_base.copy()

for source_name in (
    "tradestats",
    "obstats",
    "orderstats"
):

    feature_key = SOURCE_TO_FEATURE[source_name]

    causal = attach_causal(
        causal,
        source_name,
        validated[source_name],
        feature_frames[feature_key]
    )

    print(
        f"{source_name:12s} attached | "
        f"shape={causal.shape}"
    )


print()
print("=" * 70)
print("CAUSAL JOIN COMPLETE")
print("=" * 70)
print(f"Bars          : {len(bar_base):,}")
print(f"Causal matrix : {causal.shape}")
print("=" * 70)

CAUSAL JOIN
Bars: 5,221
Period: 2026-08-28 04:02:32+00:00 -> 2026-10-05 19:37:27+00:00

validated keys:
['tradestats', 'obstats', 'orderstats']

feature_frames keys:
['tradestats_pressure', 'orderstats_pressure', 'obstats_topology']

Resolved source → feature frame:
  tradestats   -> tradestats_pressure
  obstats      -> obstats_topology
  orderstats   -> orderstats_pressure

tradestats   attached | shape=(5221, 98)
obstats      attached | shape=(5221, 236)
orderstats   attached | shape=(5221, 409)

CAUSAL JOIN COMPLETE
Bars          : 5,221
Causal matrix : (5221, 409)


In [9]:
# =========================================================
# 11. BUILD FINAL FEATURE MATRIX
# =========================================================
#
# Никакого нового feature engineering.
#
# Используем:
#   base2  -> 73 base features
#   causal -> full Super Candle features + provenance
#
# Если base__* уже находятся в causal, повторно их НЕ join'им.
# =========================================================

print("=" * 70)
print("BUILD FINAL FEATURE MATRIX")
print("=" * 70)

# ---------------------------------------------------------
# 1. Проверка base2
# ---------------------------------------------------------

if not isinstance(base2, pd.DataFrame):
    raise RuntimeError("base2 is not a DataFrame")

BASE_COLUMNS = [
    c for c in base2.columns
    if c.startswith("base__")
]

print(f"Base features: {len(BASE_COLUMNS)}")

if len(BASE_COLUMNS) != len(BASE_FEATURES):
    raise RuntimeError(
        f"Unexpected base feature count: "
        f"{len(BASE_COLUMNS)} != {len(BASE_FEATURES)}"
    )

if "ts_utc" not in base2.columns:
    raise RuntimeError(
        "'ts_utc' not found in base2"
    )

# ---------------------------------------------------------
# 2. Проверка causal
# ---------------------------------------------------------

if not isinstance(causal, pd.DataFrame):
    raise RuntimeError("causal is not a DataFrame")

if "bar_ts" not in causal.columns:
    raise RuntimeError(
        "'bar_ts' not found in causal"
    )

if "bar_ts_utc" not in causal.columns:
    raise RuntimeError(
        "'bar_ts_utc' not found in causal"
    )

causal = causal.copy()

causal["bar_ts_utc"] = pd.to_datetime(
    causal["bar_ts_utc"],
    errors="coerce",
    utc=True
)

causal = (
    causal
    .dropna(subset=["bar_ts_utc"])
    .sort_values("bar_ts_utc")
    .drop_duplicates("bar_ts_utc", keep="last")
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# 3. Проверяем, находятся ли Base features уже в causal
# ---------------------------------------------------------

base_already_in_causal = all(
    c in causal.columns
    for c in BASE_COLUMNS
)

print(
    "Base features already in causal:",
    base_already_in_causal
)

# ---------------------------------------------------------
# 4. Если Base features отсутствуют — добавляем их
# ---------------------------------------------------------

if not base_already_in_causal:

    base_join = base2[
        ["ts_utc"] + BASE_COLUMNS
    ].copy()

    base_join["ts_utc"] = pd.to_datetime(
        base_join["ts_utc"],
        errors="coerce",
        utc=True
    )

    base_join = (
        base_join
        .dropna(subset=["ts_utc"])
        .sort_values("ts_utc")
        .drop_duplicates("ts_utc", keep="last")
    )

    causal = causal.merge(
        base_join,
        left_on="bar_ts_utc",
        right_on="ts_utc",
        how="left",
        validate="one_to_one",
        indicator="_base_merge"
    )

    missing_base = causal["_base_merge"].eq("left_only").sum()

    if missing_base:
        raise RuntimeError(
            f"Base feature alignment failed: "
            f"{missing_base:,} bars have no matching base features."
        )

    causal = causal.drop(
        columns=[
            c for c in ["_base_merge", "ts_utc"]
            if c in causal.columns
        ]
    )

    print("Base features attached to causal.")

else:
    print("Base features already present. No second join performed.")

# ---------------------------------------------------------
# 5. Все Super Candle признаки
# ---------------------------------------------------------

SOURCE_PREFIXES = (
    "tradestats__",
    "obstats__",
    "orderstats__",
)

SUPER_CANDLE_COLUMNS = [
    c for c in causal.columns
    if c.startswith(SOURCE_PREFIXES)
]

# ---------------------------------------------------------
# 6. Provenance — НЕ ML features
# ---------------------------------------------------------

PROVENANCE_COLUMNS = [
    c for c in SUPER_CANDLE_COLUMNS
    if c.endswith("__event_ts")
    or c.endswith("__available_at")
]

SUPER_CANDLE_FEATURES = [
    c for c in SUPER_CANDLE_COLUMNS
    if c not in PROVENANCE_COLUMNS
]

print(f"Super Candle features: {len(SUPER_CANDLE_FEATURES)}")
print(f"Provenance columns   : {len(PROVENANCE_COLUMNS)}")

# ---------------------------------------------------------
# 7. Полный список ML features
# ---------------------------------------------------------

FEATURE_COLUMNS = (
    BASE_COLUMNS +
    SUPER_CANDLE_FEATURES
)

# Уникальность
if len(FEATURE_COLUMNS) != len(set(FEATURE_COLUMNS)):

    duplicated = (
        pd.Series(FEATURE_COLUMNS)
        [pd.Series(FEATURE_COLUMNS).duplicated()]
        .tolist()
    )

    raise RuntimeError(
        "Duplicate feature names detected:\n"
        + "\n".join(duplicated)
    )

# ---------------------------------------------------------
# 8. Проверяем, что ВСЕ признаки реально существуют
# ---------------------------------------------------------

missing_features = [
    c for c in FEATURE_COLUMNS
    if c not in causal.columns
]

if missing_features:

    raise RuntimeError(
        "Missing final features:\n"
        + "\n".join(missing_features[:100])
    )

# ---------------------------------------------------------
# 9. Проверяем отсутствие технических конфликтов
# ---------------------------------------------------------

for bad_suffix in ("_x", "_y"):

    conflicts = [
        c for c in causal.columns
        if c.endswith(bad_suffix)
        and (
            c.startswith("base__")
            or c.startswith("tradestats__")
            or c.startswith("obstats__")
            or c.startswith("orderstats__")
        )
    ]

    if conflicts:
        raise RuntimeError(
            f"Unexpected merge conflict columns "
            f"with suffix '{bad_suffix}':\n"
            + "\n".join(conflicts[:50])
        )

# ---------------------------------------------------------
# 10. Финальная матрица
# ---------------------------------------------------------

FINAL_FEATURE_MATRIX = causal[
    [
        "bar_ts",
        "bar_ts_utc"
    ]
    + FEATURE_COLUMNS
    + PROVENANCE_COLUMNS
].copy()

# ---------------------------------------------------------
# 11. Финальная информация
# ---------------------------------------------------------

print()
print("=" * 70)
print("FINAL FEATURE MATRIX")
print("=" * 70)

print(f"Rows                  : {len(FINAL_FEATURE_MATRIX):,}")
print(f"Base features         : {len(BASE_COLUMNS):,}")
print(f"Super Candle features : {len(SUPER_CANDLE_FEATURES):,}")
print(f"TOTAL ML FEATURES     : {len(FEATURE_COLUMNS):,}")
print(f"Provenance columns    : {len(PROVENANCE_COLUMNS):,}")
print(f"Matrix shape          : {FINAL_FEATURE_MATRIX.shape}")

print("=" * 70)

BUILD FINAL FEATURE MATRIX
Base features: 73
Base features already in causal: False
Base features attached to causal.
Super Candle features: 401
Provenance columns   : 6

FINAL FEATURE MATRIX
Rows                  : 5,221
Base features         : 73
Super Candle features : 401
TOTAL ML FEATURES     : 474
Provenance columns    : 6
Matrix shape          : (5221, 482)


In [10]:
# =========================================================
# 12. FINAL DATASET HANDOFF
# =========================================================
#
# Cell 11 (above) is the ONLY place that builds
# FINAL_FEATURE_MATRIX.
#
# This cell does not rebuild the matrix. It only creates X
# for the next stage and performs final structural checks.
# =========================================================

print("=" * 70)
print("FINAL DATASET HANDOFF")
print("=" * 70)

if not isinstance(FINAL_FEATURE_MATRIX, pd.DataFrame):
    raise RuntimeError("FINAL_FEATURE_MATRIX is not a DataFrame.")

required = {"bar_ts", "bar_ts_utc", *FEATURE_COLUMNS}
missing = required - set(FINAL_FEATURE_MATRIX.columns)

if missing:
    raise RuntimeError(
        "FINAL_FEATURE_MATRIX is missing columns:\n"
        + "\n".join(sorted(missing)[:100])
    )

if len(FEATURE_COLUMNS) != (
    len(BASE_COLUMNS) + len(SUPER_CANDLE_FEATURES)
):
    raise RuntimeError(
        "Feature inventory arithmetic mismatch."
    )

if len(FEATURE_COLUMNS) != len(set(FEATURE_COLUMNS)):
    raise RuntimeError("Duplicate feature names detected.")

if set(FEATURE_COLUMNS) & set(PROVENANCE_COLUMNS):
    raise RuntimeError(
        "Provenance columns must not be ML features."
    )

X = FINAL_FEATURE_MATRIX[
    ["bar_ts"] + FEATURE_COLUMNS
].copy()

print(f"Rows          : {len(X):,}")
print(f"Base features : {len(BASE_COLUMNS):,}")
print(f"Super Candle  : {len(SUPER_CANDLE_FEATURES):,}")
print(f"ML features   : {len(FEATURE_COLUMNS):,}")
print(f"X shape       : {X.shape}")
print("=" * 70)


FINAL DATASET HANDOFF
Rows          : 5,221
Base features : 73
Super Candle  : 401
ML features   : 474
X shape       : (5221, 475)


In [11]:
# =========================================================
# 13. FINAL CAUSAL / LOOK-AHEAD AUDIT
# =========================================================

violations = []

# ---------------------------------------------------------
# A. Проверяем provenance всех трех Super Candle источников
# ---------------------------------------------------------

for source in ("tradestats", "obstats", "orderstats"):

    av_col = f"{source}__available_at"
    ev_col = f"{source}__event_ts"

    if av_col not in causal.columns:
        raise RuntimeError(
            f"{source}: missing provenance column {av_col}"
        )

    if ev_col not in causal.columns:
        raise RuntimeError(
            f"{source}: missing provenance column {ev_col}"
        )

    av = pd.to_datetime(
        causal[av_col],
        errors="coerce",
        utc=True
    )

    ev = pd.to_datetime(
        causal[ev_col],
        errors="coerce",
        utc=True
    )

    bad_av = av.notna() & (
        av > causal["bar_ts_utc"]
    )

    bad_ev = ev.notna() & (
        ev > causal["bar_ts_utc"]
    )

    violations.append({
        "source": source,
        "bars_checked": len(causal),
        "availability_violations": int(bad_av.sum()),
        "event_ts_future_violations": int(bad_ev.sum()),
        "availability_violation_pct":
            float(bad_av.mean() * 100),
        "event_future_pct":
            float(bad_ev.mean() * 100),
    })


audit_report = pd.DataFrame(violations)

display(audit_report)


# ---------------------------------------------------------
# B. Проверяем monotonic availability
# ---------------------------------------------------------

for source in ("tradestats", "obstats", "orderstats"):

    av = pd.to_datetime(
        causal[f"{source}__available_at"],
        errors="coerce",
        utc=True
    ).dropna()

    if not av.is_monotonic_increasing:
        raise RuntimeError(
            f"{source}: selected availability timestamps "
            f"are not monotonic."
        )


# ---------------------------------------------------------
# C. Проверяем базовые feature builders
#    на очевидные операции, создающие look-ahead
# ---------------------------------------------------------

import inspect

builder_source = (
    inspect.getsource(FeatureBuilder)
    + inspect.getsource(MicroFeatures)
)

forbidden_tokens = [
    "bfill(",
    ".bfill(",
    "backfill(",
    "shift(-"
]

found = [
    token
    for token in forbidden_tokens
    if token in builder_source
]

if found:
    raise RuntimeError(
        "🚨 Base-feature builder contains potentially "
        "non-causal operation(s): "
        + ", ".join(found)
    )


# ---------------------------------------------------------
# D. Проверяем полный фактический feature inventory
# ---------------------------------------------------------
#
# НИКАКОГО EXPECTED_FEATURES = 475.
#
# Мы проверяем именно тот полный набор, который реально
# собран pipeline.
# ---------------------------------------------------------

actual_feature_count = len(FEATURE_COLUMNS)

if actual_feature_count != (
    len(BASE_COLUMNS) + len(SUPER_CANDLE_FEATURES)
):
    raise RuntimeError(
        "Feature inventory arithmetic mismatch."
    )


if len(FEATURE_COLUMNS) != len(
    set(FEATURE_COLUMNS)
):
    duplicates = (
        pd.Series(FEATURE_COLUMNS)
        [pd.Series(FEATURE_COLUMNS).duplicated()]
        .tolist()
    )

    raise RuntimeError(
        "Duplicate feature names detected:\n"
        + "\n".join(duplicates)
    )


# ---------------------------------------------------------
# E. Проверяем отсутствие look-ahead
# ---------------------------------------------------------

if (
    audit_report["availability_violations"] > 0
).any():

    raise RuntimeError(
        "🚨 LOOK-AHEAD DETECTED: "
        "a feature row was used before availability."
    )


if (
    audit_report["event_ts_future_violations"] > 0
).any():

    raise RuntimeError(
        "🚨 LOOK-AHEAD DETECTED: "
        "a source event timestamp is in the future."
    )


# ---------------------------------------------------------
# F. Финальный результат
# ---------------------------------------------------------

print("=" * 90)
print("FINAL CAUSAL CHECK: PASS")
print("=" * 90)

print(f"Rows checked       : {len(causal):,}")
print(f"ML features        : {len(FEATURE_COLUMNS)}")
print(f"Base features      : {len(BASE_COLUMNS)}")
print(f"Super Candle       : {len(SUPER_CANDLE_FEATURES)}")
print(f"Provenance columns : {len(PROVENANCE_COLUMNS)}")

print()
print("Availability rule  : available_at <= bar_ts")
print("Event rule         : event_ts <= bar_ts")
print("Join direction     : backward")
print("Base builders      : no bfill/backfill/negative shift detected")
print("Orderbook          : excluded from historical Super Candle matrix")

print("=" * 90)

,source,bars_checked,availability_violations,event_ts_future_violations,availability_violation_pct,event_future_pct
0,tradestats,5221,0,0,0.0,0.0
1,obstats,5221,0,0,0.0,0.0
2,orderstats,5221,0,0,0.0,0.0


FINAL CAUSAL CHECK: PASS
Rows checked       : 5,221
ML features        : 474
Base features      : 73
Super Candle       : 401
Provenance columns : 6

Availability rule  : available_at <= bar_ts
Event rule         : event_ts <= bar_ts
Join direction     : backward
Base builders      : no bfill/backfill/negative shift detected
Orderbook          : excluded from historical Super Candle matrix


In [12]:
# =========================
# 14. QUALITY / PROVENANCE SUMMARY
# =========================

print("Ticker:", TICKER)
print("Bars:", len(X))
print("Feature count:", len(FEATURE_COLUMNS))
print("Orderbook included:", False)

print("\nFeature groups:")
print("base_73        =", len(BASE_COLUMNS))
print("tradestats_94  =", len(feature_groups["tradestats_pressure"]))
print("orderstats_171 =", len(feature_groups["orderstats_pressure"]))
print("obstats_136    =", len(feature_groups["obstats_topology"]))
print("Super Candle   =", len(SUPER_CANDLE_FEATURES))
print("TOTAL ML       =", len(FEATURE_COLUMNS))

print("\nMissingness top 20:")
display(
    X[FEATURE_COLUMNS]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .head(20)
    .to_frame("missing_pct")
)

# Final handoff object for the future model notebook
FINAL_DATASET = X.copy()

print("\nFINAL_DATASET shape:", FINAL_DATASET.shape)
print("Final dataset is ready for the next stage: labels / temporal split / model.")


Ticker: VTBR
Bars: 5221
Feature count: 474
Orderbook included: False

Feature groups:
base_73        = 73
tradestats_94  = 94
orderstats_171 = 171
obstats_136    = 136
Super Candle   = 401
TOTAL ML       = 474

Missingness top 20:


,missing_pct
tradestats__ts_flow_x_return_ret,0.065696
tradestats__ts_vwap_b_s_bps_ret,0.021069
obstats__ob_vol_imb_bbo_vs_deep_ret,0.014365
obstats__ob_val_imb_bbo_vs_deep_ret,0.012641
tradestats__ts_trade_imbalance_ret,0.005171
orderstats__os_put_orders_imb_ret,0.001341
orderstats__os_cancel_orders_imb_ret,0.001341
obstats__ob_1mio_liquidity_cost_bps_z6,0.000958
obstats__ob_1mio_to_bbo_spread_ratio_z24,0.000958
obstats__ob_1mio_to_bbo_spread_ratio_z12,0.000958



FINAL_DATASET shape: (5221, 475)
Final dataset is ready for the next stage: labels / temporal split / model.
